# Task 2: Unemployment Analysis with Python
Objective: Perform exploratory data analysis on unemployment data to uncover regional and temporal trends, with a focus on the impact of the COVID-19 pandemic on unemployment rates in India.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Load the Dataset
Please ensure `Unemployment_in_India.csv` is in the same directory. You can download it from Kaggle.

In [ ]:
try:
    df = pd.read_csv("Unemployment_in_India.csv")
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: Please download 'Unemployment_in_India.csv' from Kaggle and place it in the same directory.")
    # Creating dummy data for demonstration purposes if file is missing
    dates = pd.date_range(start="2019-01-01", end="2020-12-31", freq="M")
    regions = ["Andhra Pradesh", "Maharashtra", "Uttar Pradesh", "Kerala", "Haryana"]
    data = []
    for r in regions:
        for d in dates:
            data.append([r, d.strftime("%d-%m-%Y"), "Monthly", np.random.uniform(5, 25), np.random.uniform(100000, 500000), np.random.uniform(35, 55)])
    df = pd.DataFrame(data, columns=["Region", " Date", " Frequency", " Estimated Unemployment Rate (%)", " Estimated Employed", " Estimated Labour Participation Rate (%)"])

### Data Inspection and Cleaning

In [ ]:
df.columns = df.columns.str.strip() # Strip whitespace from column names
print("Shape:", df.shape)
print("\nNull Values:\n", df.isnull().sum())
df = df.dropna()
print("\nData Types:\n", df.dtypes)

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
print("\nData Types after conversion:\n", df.dtypes)
df.head()

### EDA: Region-wise Average Unemployment Rates

In [ ]:
region_stats = df.groupby("Region")["Estimated Unemployment Rate (%)"].mean().reset_index()
region_stats = region_stats.sort_values(by="Estimated Unemployment Rate (%)", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=region_stats, x="Region", y="Estimated Unemployment Rate (%)")
plt.xticks(rotation=90)
plt.title("Average Unemployment Rate by Region")
plt.show()

### EDA: Month-wise Trends

In [ ]:
df["Month_Year"] = df["Date"].dt.to_period("M")
monthly_stats = df.groupby("Month_Year")["Estimated Unemployment Rate (%)"].mean().reset_index()
monthly_stats["Month_Year"] = monthly_stats["Month_Year"].astype(str)

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_stats, x="Month_Year", y="Estimated Unemployment Rate (%)", marker="o")
plt.xticks(rotation=45)
plt.title("Month-wise Average Unemployment Rate")
plt.show()

### Time-Series Line Chart: 3 Major States

In [ ]:
major_states = ["Maharashtra", "Uttar Pradesh", "Andhra Pradesh"]
subset_df = df[df["Region"].isin(major_states)]

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset_df, x="Date", y="Estimated Unemployment Rate (%)", hue="Region", marker="o")
plt.title("Unemployment Rate Over Time for Major States")
plt.show()

### Bar Chart: Top 10 States with Highest Average Unemployment Rate

In [ ]:
top_10 = region_stats.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(data=top_10, x="Estimated Unemployment Rate (%)", y="Region", palette="Reds_r")
plt.title("Top 10 States with Highest Average Unemployment Rate")
plt.show()

### Heatmap: Correlation

In [ ]:
corr_cols = ["Estimated Unemployment Rate (%)", "Estimated Employed", "Estimated Labour Participation Rate (%)"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

The heatmap shows the correlation between unemployment rate, employment numbers, and labor participation. Typically, employment is negatively correlated with unemployment.

### Pre-COVID vs Post-COVID Comparison

In [ ]:
# Assuming March 2020 as the cutoff for Pre/Post COVID (lockdown started late March)
df["COVID_Era"] = np.where(df["Date"] < "2020-03-01", "Pre-COVID", "Post-COVID")

covid_stats = df.groupby("COVID_Era")["Estimated Unemployment Rate (%)"].mean().reset_index()
display(covid_stats)

plt.figure(figsize=(6, 5))
sns.barplot(data=covid_stats, x="COVID_Era", y="Estimated Unemployment Rate (%)")
plt.title("Pre-COVID vs Post-COVID Unemployment Rate")
plt.show()

We observe a significant spike in the unemployment rate during the Post-COVID era, correlating strongly with nationwide lockdowns.